# Imports and Setup

In [1]:
import os
import io
import sys
import time
import pandas as pd
from tqdm import tqdm
import anthropic
from concurrent.futures import ProcessPoolExecutor
from dotenv import load_dotenv
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import DeepEvalBaseLLM

/home/eqp6pg/.conda/envs/my-torch/lib/python3.8/site-packages/deepeval/__init__.py:51: UserWarning: You are using deepeval version 2.0, however version 2.7.0 is available. You should consider upgrading via the "pip install --upgrade deepeval" command.
  warnings.warn(


In [2]:
os.cpu_count()

96

# Claude connection

In [3]:
load_dotenv()
anthropic_key = os.getenv("ANTHROPIC_KEY")

In [4]:
# Define a custom Claude model for deepeval
class Claude(DeepEvalBaseLLM):
    """Class to implement Claude model for DeepEval"""
    def __init__(self, model, api_key):
        self.model = model
        self.api_key = api_key
        self.client = anthropic.Anthropic(api_key=self.api_key)
    
    def load_model(self):
        return self.model
    
    def generate(self, prompt, max_tokens=4096):
        response = self.client.messages.create(
            model=self.model,
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text
    
    async def a_generate(self, prompt, max_tokens=4096):
        response = self.client.messages.create(
            model=self.model,
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text
    
    def get_model_name(self):
        return "Claude Model"

In [5]:
claude_model = "claude-3-haiku-20240307"
claude_instance = Claude(model=claude_model, api_key=anthropic_key)

# Dataset

In [22]:
df = pd.read_csv("data/data.csv")

# Create Deepeval test cases
test_cases = [
    LLMTestCase(
        input=row["input"],
        actual_output=row["actual_output"]
    )
    for _, row in df.iterrows()
]

In [23]:
len(test_cases)

1600

In [24]:
test_cases = test_cases[1000:1600]

# Evaluation

In [25]:
# Function to evaluate a single case with error handling
def evaluate_case_safe(test_case, index):
    try:
        # Suppress output for each worker process
        original_stdout = sys.stdout
        sys.stdout = io.StringIO()
        
        # Using the single case functionality
        summarization_metric = SummarizationMetric(
            model=claude_instance,
            include_reason=False
        )
        
        summarization_metric.measure(test_case)
        score = summarization_metric.score
        score_bd = summarization_metric.score_breakdown
        
        # Restore stdout
        sys.stdout = original_stdout
        
        return {
            "index": index,
            "input": test_case.input[:10] + "...",
            "score": score,
            "score_bd": score_bd,
            "success": True
        }
    except Exception as e:
        # Restore stdout on error
        sys.stdout = original_stdout
        return {
            "index": index,
            "input": test_case.input[:10] + "...",
            "score": None,
            "score_bd": None,
            "success": False,
            "error": str(e)
        }

In [26]:
# Start timer
start_time = time.time()

# Get available CPU cores and set max workers
cpu_cores = os.cpu_count()
max_workers = min(cpu_cores, 10)  # Ensure we don't exceed available cores

# Suppress main process output
original_stdout = sys.stdout
sys.stdout = io.StringIO()

# Parallel evaluation with tqdm and error handling
results = []
with ProcessPoolExecutor(max_workers=max_workers) as executor:
    futures = {executor.submit(evaluate_case_safe, test_cases[i], i): i for i in range(len(test_cases))}
    
    # Restore stdout for progress bar
    sys.stdout = original_stdout
    
    for future in tqdm(futures, total=len(test_cases), desc="Evaluating Summarization"):
        result = future.result()
        results.append(result)

# End timer
end_time = time.time()
elapsed_time = end_time - start_time

Evaluating Summarization: 100%|██████████| 600/600 [09:05<00:00,  1.10it/s]


In [27]:
# Sort results by index to maintain original order
results.sort(key=lambda x: x["index"])

# Create a DataFrame from results and save to CSV
results_df = pd.DataFrame(results)
results_df.to_csv("summarization_scores_1000-1600.csv", index=False)

In [28]:
# Print summary
print(f"\nEvaluation completed in {elapsed_time:.2f} seconds")
print(f"Processed {len(test_cases)} test cases")
print(f"Successful evaluations: {sum(1 for r in results if r['success'])}")
print(f"Failed evaluations: {sum(1 for r in results if not r['success'])}")
print(f"Results saved to: summarization_scores.csv")


Evaluation completed in 545.83 seconds
Processed 600 test cases
Successful evaluations: 533
Failed evaluations: 67
Results saved to: summarization_scores.csv


In [29]:
# Print a sample of scores if available
successful_scores = [r["score"] for r in results if r["success"] and r["score"] is not None]
if successful_scores:
    print(f"\nScore statistics:")
    print(f"  Min score: {min(successful_scores):.4f}")
    print(f"  Max score: {max(successful_scores):.4f}")
    print(f"  Avg score: {sum(successful_scores)/len(successful_scores):.4f}")


Score statistics:
  Min score: 0.0000
  Max score: 1.0000
  Avg score: 0.5396
